# Customer Insights & Analytics Dashboard - Group 5

### Members :
- Saad Saeed Almalki

### Supervised by :
- Omar Hejazi
- Taraf Almanjomi

---
**Taif University - DSC Google , Data Science and Engineering**

#### Using Pandas :

In [4]:
import pandas as pd
import sqlite3

#### Load data :

In [7]:
customers  = pd.read_csv("01_Customers.csv")
products   = pd.read_csv("02_Products.csv")
suppliers  = pd.read_csv("03_Suppliers.csv")
warehouses = pd.read_csv("04_Warehouses.csv")
orders     = pd.read_csv("05_Orders.csv")
order_items = pd.read_csv("06_Order_Items.csv")
reviews    = pd.read_csv("07_Reviews.csv")
inventory  = pd.read_csv("08_Inventory.csv")

In [9]:
tables = {
    "Customers": customers,
    "Products": products,
    "Suppliers": suppliers,
    "Warehouses": warehouses,
    "Orders": orders,
    "Order_Items": order_items,
    "Reviews": reviews,
    "Inventory": inventory
}

for name, df in tables.items():
    print(f" rows : {name}: {len(df)}")

 rows : Customers: 503
 rows : Products: 27
 rows : Suppliers: 12
 rows : Warehouses: 5
 rows : Orders: 1400
 rows : Order_Items: 2583
 rows : Reviews: 854
 rows : Inventory: 135


#### Check data :

In [12]:
customers.head()

,customer_id,customer_name,gender,age,city,region,signup_date,customer_segment
0,C0001,Customer 0001,Female,65.0,Al Kharj,Central,2024-09-07,New
1,C0002,Customer 0002,Prefer not to say,65.0,Majmaah,Central,2024-03-30,Regular
2,C0003,Customer 0003,Female,23.0,Riyadh,Central,2024-08-26,Regular
3,C0004,Customer 0004,Prefer not to say,30.0,Majmaah,Central,2025-07-12,Regular
4,C0005,Customer 0005,Prefer not to say,35.0,Abha,Southern,2024-06-12,Regular


#### Combine all CSV files into one excel sheet :

In [15]:
with pd.ExcelWriter("Data.xlsx") as writer:
    
    for name, df in tables.items():
        
        df.to_excel(writer, sheet_name=name, index=False)

#### Data cleaning :

In [18]:
cleaned = {}


for name, df in tables.items():
    cleaned[name] = df.copy()

##### Remove duplicate values :

In [21]:
for df in cleaned.values():
    text_columns = df.select_dtypes(include="object").columns

    for column in text_columns:
        
        df[column] = df[column].str.strip().replace("", pd.NA)

In [23]:
for name, df in cleaned.items():
    duplicates = df.duplicated().sum()
    
    cleaned[name] = df.drop_duplicates().reset_index(drop=True)

    
    print(f"{name}: deleted {duplicates} duplicate rows")


Customers: deleted 3 duplicate rows
Products: deleted 0 duplicate rows
Suppliers: deleted 0 duplicate rows
Warehouses: deleted 0 duplicate rows
Orders: deleted 0 duplicate rows
Order_Items: deleted 0 duplicate rows
Reviews: deleted 3 duplicate rows
Inventory: deleted 0 duplicate rows


In [25]:
reviews = cleaned["Reviews"]
order_items = cleaned["Order_Items"]
inventory = cleaned["Inventory"]



bad_ratings = reviews["rating"].notna() & ~reviews["rating"].between(1, 5)
badquantities = order_items["quantity"] <= 0

bad_stock = inventory["stock_on_hand"] < 0

reviews.loc[bad_ratings, "rating"] = pd.NA
order_items.loc[badquantities, "quantity"] = pd.NA
inventory.loc[bad_stock, "stock_on_hand"] = pd.NA


print(f"Incorrect ratings: {bad_ratings.sum()}")

print(f"Bad quantities: {badquantities.sum()}")

print(f"Bad stock : {bad_stock.sum()}")

Incorrect ratings: 3
Bad quantities: 4
Bad stock : 3


In [27]:
reviews = cleaned["Reviews"]
unknownorders = ~reviews["order_id"].isin(cleaned["Orders"]["order_id"])
cleaned["Reviews"] = reviews.loc[~unknownorders].copy()

items = cleaned["Order_Items"]


unknown_products = ~items["product_id"].isin(cleaned["Products"]["product_id"])
items.loc[unknown_products, "product_id"] = pd.NA

In [29]:
print(f"Unknown orders : {unknownorders.sum()}")
print(f"Unknows products : {unknown_products.sum()}")

Unknown orders : 1
Unknows products : 1


In [31]:
cleaned["Customers"]["signup_date"] = pd.to_datetime(cleaned["Customers"]["signup_date"], errors="coerce")
cleaned["Orders"]["order_date"] = pd.to_datetime(cleaned["Orders"]["order_date"], errors="coerce")
cleaned["Reviews"]["review_date"] = pd.to_datetime(cleaned["Reviews"]["review_date"], errors="coerce")
cleaned["Inventory"]["snapshot_date"] = pd.to_datetime(cleaned["Inventory"]["snapshot_date"], errors="coerce")

##### Review dates check :


In [34]:
order_dates = cleaned["Orders"].set_index("order_id")["order_date"]
reviews = cleaned["Reviews"]
real_order_dates = reviews["order_id"].map(order_dates)

baddates = reviews["review_date"] < real_order_dates.dt.normalize()
reviews_to_check = reviews.loc[baddates].copy()
reviews_to_check["order_date"] = real_order_dates.loc[baddates]

reviews.loc[baddates, "review_date"] = pd.NaT
print(f"Review dates to check: {baddates.sum()}")


Review dates to check: 424


##### Fill missing values :

In [37]:
items = cleaned["Order_Items"]
product_cost = cleaned["Products"].set_index("product_id")["unit_cost"]
unit_cost = items["product_id"].map(product_cost)

missingqty = items["quantity"].isna()
qty_guess = (items["line_cost"] / unit_cost).round()
validqty = missingqty & qty_guess.between(1, 20) & ((qty_guess * unit_cost - items["line_cost"]).abs() < 0.02)
items["quantity_filled"] = validqty
items.loc[validqty, "quantity"] = qty_guess[validqty]

missingdiscount = items["discount_pct"].isna()
discount_guess = (1 - items["line_revenue"] / (items["quantity"] * items["unit_price"])).round(2)
validdiscount = (missingdiscount & discount_guess.between(0, 1) &
                 ((items["quantity"] * items["unit_price"] * (1 - discount_guess) - items["line_revenue"]).abs() < 0.03))
items["discount_filled"] = validdiscount
items.loc[validdiscount, "discount_pct"] = discount_guess[validdiscount]

items["product_id_filled"] = False
for i in items.index[items["product_id"].isna()]:
    row = items.loc[i]
    products_match = cleaned["Products"].loc[
        (cleaned["Products"]["unit_price"] - row["unit_price"]).abs() < 0.01
    ]
    products_match = products_match.loc[
        (products_match["unit_cost"] * row["quantity"] - row["line_cost"]).abs() < 0.02
    ]
    if len(products_match) == 1:
        items.at[i, "product_id"] = products_match.iloc[0]["product_id"]
        items.at[i, "product_id_filled"] = True

print("Quantities recovered:", int(validqty.sum()))
print("Discounts recovered:", int(validdiscount.sum()))
print("Products recovered:", int(items["product_id_filled"].sum()))

Quantities recovered: 9
Discounts recovered: 6
Products recovered: 1


In [39]:

customers = cleaned["Customers"]
orders = cleaned["Orders"]
reviews = cleaned["Reviews"]
inventory = cleaned["Inventory"]

customers["age_filled"] = customers["age"].isna()
age_by_region = customers.groupby("region")["age"].transform("median").round()
customers["age"] = customers["age"].fillna(age_by_region)
customers["city"] = customers["city"].fillna("Unknown")

orders["campaign"] = orders["campaign"].fillna("Not specified")
orders["delivery_days_filled"] = orders["delivery_days"].isna() & orders["order_status"].ne("Cancelled")
days_by_status = orders.groupby("order_status")["delivery_days"].transform("median").round()
orders.loc[orders["delivery_days_filled"], "delivery_days"] = days_by_status[orders["delivery_days_filled"]]

reviews["rating_filled"] = reviews["rating"].isna()
rating_by_sentiment = reviews.groupby("sentiment_label")["rating"].transform("median").round()
reviews["rating"] = reviews["rating"].fillna(rating_by_sentiment)
reviews["review_text"] = reviews["review_text"].fillna("No text provided")
reviews["review_date_invalid"] = reviews["review_id"].isin(reviews_to_check["review_id"])

inventory["stock_filled"] = inventory["stock_on_hand"].isna()
stock_by_product = inventory.groupby("product_id")["stock_on_hand"].transform("median").round()
inventory["stock_on_hand"] = inventory["stock_on_hand"].fillna(stock_by_product)

print("Estimated ages:", int(customers["age_filled"].sum()))
print("Estimated delivery days:", int(orders["delivery_days_filled"].sum()))
print("Estimated ratings:", int(reviews["rating_filled"].sum()))
print("Estimated stock:", int(inventory["stock_filled"].sum()))
print("Some dates, cancelled deliveries and unknown customer IDs stay blank on purpose.")

Estimated ages: 5
Estimated delivery days: 8
Estimated ratings: 11
Estimated stock: 10
Some dates, cancelled deliveries and unknown customer IDs stay blank on purpose.


##### Values need to reviewed :

In [42]:
items = cleaned["Order_Items"]
calculated = items["quantity"] * items["unit_price"] * (1 - items["discount_pct"])

mismatch = calculated.notna() & ((items["line_revenue"] - calculated).abs() > 0.06)
revenue_to_check = items.loc[mismatch].copy()
revenue_to_check["calculated_revenue"] = calculated.loc[mismatch].round(2)

print(f"Revenue rows to review: {len(revenue_to_check)}")
display(revenue_to_check.head())

Revenue rows to review: 6


,order_item_id,order_id,product_id,quantity,unit_price,discount_pct,line_revenue,line_cost,quantity_filled,discount_filled,product_id_filled,calculated_revenue
295,OI00161_3,O00161,P003,1.0,60.47,0.95,54.42,40.70,False,False,False,3.02
366,OI00206_1,O00206,P013,1.0,32.01,0.95,28.81,20.31,False,False,False,1.60
776,OI00421_2,O00421,P025,2.0,1672.24,0.15,710.70,461.78,False,False,False,2842.81
1435,OI00770_3,O00770,P015,1.0,905.48,0.00,226.37,144.70,False,False,False,905.48
1530,OI00823_3,O00823,P013,1.0,128.04,0.20,25.61,20.31,False,False,False,102.43


In [44]:
for name, df in cleaned.items():
    missing_count = df.isna().sum().sum()
    print(f"{name}Missing Counts  : {missing_count}")

CustomersMissing Counts  : 0
ProductsMissing Counts  : 0
SuppliersMissing Counts  : 0
WarehousesMissing Counts  : 0
OrdersMissing Counts  : 101
Order_ItemsMissing Counts  : 0
ReviewsMissing Counts  : 424
InventoryMissing Counts  : 0


#### Cleaned excel sheet :

In [47]:
with pd.ExcelWriter("Datacleaned.xlsx") as writer:
    for name, df in cleaned.items():
        df.to_excel(writer, sheet_name=name, index=False)

    revenue_to_check.to_excel(writer, sheet_name="Revenue_To_Check", index=False)
    reviews_to_check.to_excel(writer, sheet_name="Reviews_To_Check", index=False)


#### SQL queris

In [50]:
conn = sqlite3.connect("DataToImpact.db")
for name, df in cleaned.items():
    df.to_sql(name, conn, if_exists="replace", index=False)

In [52]:
query = """
SELECT order_status, COUNT(*) AS total_orders
FROM Orders
GROUP BY order_status
ORDER BY total_orders DESC
"""
display(pd.read_sql_query(query, conn))

,order_status,total_orders
0,Delivered,1102
1,Returned,128
2,Cancelled,97
3,Processing,73


##### Best-selling products by number of items.

In [55]:
query = """
SELECT p.product_name, SUM(i.quantity) AS units_sold
FROM Order_Items AS i
JOIN Products AS p ON i.product_id = p.product_id
GROUP BY p.product_id, p.product_name
ORDER BY units_sold DESC
LIMIT 10
"""
display(pd.read_sql_query(query, conn))

,product_name,units_sold
0,Keyboard,245.0
1,Sunscreen,229.0
2,Water Bottle,221.0
3,Storage Box,211.0
4,Hair Dryer,201.0
5,Perfume,201.0
6,Smart Watch,198.0
7,Phone Charger,197.0
8,Sneakers,197.0
9,Moisturizer,193.0


##### Top customers by total spent

In [58]:
query = """
SELECT c.customer_name,
       COUNT(DISTINCT o.order_id) AS orders_count,
       ROUND(SUM(i.line_revenue), 2) AS total_spent
FROM Customers c
JOIN Orders o ON c.customer_id = o.customer_id
JOIN Order_Items i ON o.order_id = i.order_id
WHERE o.order_status = 'Delivered'
GROUP BY c.customer_id, c.customer_name
ORDER BY total_spent DESC
LIMIT 10
"""

display(pd.read_sql_query(query, conn))

,customer_name,orders_count,total_spent
0,Customer 0121,8,6066.75
1,Customer 0182,4,5861.04
2,Customer 0105,5,5700.49
3,Customer 0270,3,5651.69
4,Customer 0470,7,5512.47
5,Customer 0441,5,5485.44
6,Customer 0313,5,5375.44
7,Customer 0005,8,5033.71
8,Customer 0215,9,4938.58
9,Customer 0315,7,4808.68


##### Low stock products :

In [61]:
query = """
SELECT w.warehouse_name,
       p.product_name,
       i.stock_on_hand,
       i.reorder_level
FROM Inventory i
JOIN Warehouses w ON i.warehouse_id = w.warehouse_id
JOIN Products p ON i.product_id = p.product_id
WHERE i.stock_on_hand <= i.reorder_level
  AND i.stock_filled = 0
ORDER BY i.stock_on_hand ASC
"""
display(pd.read_sql_query(query, conn))

,warehouse_name,product_name,stock_on_hand,reorder_level
0,Jeddah Warehouse,Bedsheet Set,2.0,37
1,Tabuk Warehouse,Mouse,2.0,32
2,Tabuk Warehouse,Moisturizer,4.0,28
3,Riyadh Central Warehouse,Coffee Maker,5.0,41
4,Dammam Warehouse,Air Fryer,5.0,60
5,Tabuk Warehouse,Bluetooth Speaker,5.0,20
6,Jeddah Warehouse,Mouse,7.0,48
7,Abha Warehouse,Bedsheet Set,9.0,21
8,Abha Warehouse,Phone Charger,10.0,60
9,Riyadh Central Warehouse,Moisturizer,15.0,59


##### Delivery Performance of warehouses

In [64]:
query = """
SELECT w.warehouse_name,
       COUNT(*) AS delivered_orders,
       ROUND(AVG(o.delivery_days), 2) AS avg_delivery_days
FROM Orders o
JOIN Warehouses w ON o.warehouse_id = w.warehouse_id
WHERE o.order_status = 'Delivered'
  AND o.delivery_days IS NOT NULL
  AND o.delivery_days_filled = 0
GROUP BY w.warehouse_name
ORDER BY avg_delivery_days DESC
"""


display(pd.read_sql_query(query, conn))

,warehouse_name,delivered_orders,avg_delivery_days
0,Riyadh Central Warehouse,226,4.63
1,Jeddah Warehouse,211,4.57
2,Abha Warehouse,207,4.55
3,Tabuk Warehouse,246,4.51
4,Dammam Warehouse,206,4.43


In [66]:
import sqlite3

conn = sqlite3.connect("DataToImpact.db")

with open("DataToImpact.sql", "w", encoding="utf-8") as file:
    for line in conn.iterdump():
        file.write(line + "\n")

conn.close()

#### Text Analysis and Pattern Detection :

##### Most repeated customer comments :

In [70]:
reviews = cleaned["Reviews"]

comments = reviews["review_text"].replace("No text provided", pd.NA).dropna()

print("Reviews with text:", len(comments))
display(comments.value_counts().head(10).rename_axis("Review").reset_index(name="Count"))

Reviews with text: 844


,Review,Count
0,التغليف ممتاز لكن السعر مرتفع,53
1,جودة ممتازة وسأكرر الشراء,53
2,تجربة شراء سهلة وسريعة,49
3,التوصيل كان سريع والمنتج ممتاز,47
4,جودة المنتج أقل من المتوقع,44
5,وصل الطلب متأخر والتغليف كان سيئ,43
6,المنتج مطابق للوصف وأنصح فيه,42
7,المنتج لا يعمل بشكل صحيح,40
8,السعر مناسب والتجربة جميلة,39
9,خدمة العملاء ممتازة,39


##### Common customer complaints (Arabic and English) :

In [73]:
issues = {
    "Shipping delays": r"متأخر|تأخر|delayed|arrived late",
    "Product issues": r"أقل من المتوقع|لا يعمل|stopped working|lower than expected",
    "High prices": r"السعر مرتفع|price is high",
    "Customer service": r"لم ترد|did not respond|no response",
    "Size / replacement": r"المقاس غير مناسب|الاستبدال|replacement",
    "Damaged packaging": r"التغليف كان سيئ|packaging was damaged"
}

In [75]:
problem_counts = []
for name, keywords in issues.items():
    total = comments.str.contains(keywords, case=False, regex=True, na=False).sum()
    problem_counts.append({"Problem": name, "Mentions": int(total)})


problems = pd.DataFrame(problem_counts).sort_values("Mentions", ascending=False)

In [77]:
display(problems)

,Problem,Mentions
1,Product issues,152
0,Shipping delays,132
2,High prices,86
5,Damaged packaging,66
4,Size / replacement,37
3,Customer service,35
